### Database Connection 

In [1]:
import pyodbc
import pandas as pd 
import numpy as np

DRIVER_NAME = 'ODBC Driver 17 for SQL Server'
SERVER_NAME = r'DESKTOP-L3GBMQ5\SQLEXPRESS'
DATABASE_NAME = 'Financial_CaseStudy'

connection_string = (
    f"DRIVER={{{DRIVER_NAME}}};"
    f"SERVER={SERVER_NAME};"
    f"DATABASE={DATABASE_NAME};"
    f"Trusted_Connection=yes;"
)

conn = pyodbc.connect(connection_string)
cursor = conn.cursor()
print("Connected successfully!")

Connected successfully!


### Load Data

In [2]:
query_customers = "SELECT * FROM stg.stg_dim_Customers"
query_loan = "SELECT * FROM stg.stg_Loan_applications"
query_payments = "SELECT * FROM stg.stg_fact_payments"

In [3]:
df_customers = pd.read_sql(query_customers, conn)
df_loan = pd.read_sql(query_loan, conn)
df_payments = pd.read_sql(query_payments, conn)

C:\Users\GIGABYTE\AppData\Local\Temp\ipykernel_8812\529905624.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_customers = pd.read_sql(query_customers, conn)
C:\Users\GIGABYTE\AppData\Local\Temp\ipykernel_8812\529905624.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_loan = pd.read_sql(query_loan, conn)
C:\Users\GIGABYTE\AppData\Local\Temp\ipykernel_8812\529905624.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_payments = pd.read_sql(query_payments, conn)


### Missing values Summry

In [5]:
def missing_summary(df):
    return df.isnull().sum().to_frame('Missing_Count').assign(
        Missing_Percentage=lambda x: (x['Missing_Count'] / len(df)) * 100
    )

In [7]:
print("\n Missing Values Overview:")
print("\nCustomers Table:\n", missing_summary(df_customers))
print("\nLoans Table:\n", missing_summary(df_loan))
print("\nPayments Table:\n", missing_summary(df_payments))


 Missing Values Overview:

Customers Table:
                        Missing_Count  Missing_Percentage
Customer_ID                        0                 0.0
Full_Name                          0                 0.0
Gender                             0                 0.0
Date_of_Birth                      0                 0.0
Region                             0                 0.0
Education_Level                    0                 0.0
Employment_Status                  0                 0.0
Annual_Income                      0                 0.0
Credit_History_Length              0                 0.0
Credit_Score                       0                 0.0

Loans Table:
                    Missing_Count  Missing_Percentage
Loan_ID                        0                 0.0
Customer_ID                    0                 0.0
Loan_Amount                    0                 0.0
Loan_Term_Months               0                 0.0
Interest_Rate                  0               

### Relationship Validation

In [9]:
# Check if all Customer_ID in loan_applications exist in dim_Customers
cust_mismatch = df_loan[~df_loan['Customer_ID'].isin(df_customers['Customer_ID'])]
print(f"\n loan records without matching customers : {len(cust_mismatch)}")


 loan records without matching customers : 0


In [11]:
# Check if all Loan_IDs in fact_payments exist in loan_applications
loan_mismatch = df_payments[~df_payments['Loan_ID'].isin(df_loan['Loan_ID'])]
print(f" Payment records without matching loans: {len(loan_mismatch)}")

 Payment records without matching loans: 0


In [12]:
conn.close()